<a href="https://colab.research.google.com/github/K-Pratham-Prabhu09/Prototype-Lab/blob/pytorch_tutorials/Pytorch_practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

### Level 1: The Noisy Sensor Array (Signal Processing & Safe Cloning)
You are analyzing high-frequency data from a phased array radar system. The data is incredibly noisy, and you need to isolate the anomalies (potential targets) without corrupting your original raw data.

**The Scenario:**
You have 5 sensors recording data for 1,000 milliseconds.

**Your Mission:**
1. **The Raw Feed:** Create a tensor `raw_signal` of shape `(1000, 5)` filled with random floats from a standard normal distribution (`torch.randn`).
2. **The Golden Rule of Backups:** You need to process this data, but changing `raw_signal` directly will destroy the original feed. Create a completely independent copy named `processed_signal`. (Do *not* just use `=`).
3. **Global Extrema:** Find the absolute maximum value across the *entire* tensor and its exact index using `torch.argmax()`.
4. **Statistical Baselining:** Calculate the `mean` and standard deviation (`std`) for *each* of the 5 sensors. You should get two tensors, each of shape `(5,)`.
5. **Anomaly Detection (Boolean Masks):** An anomaly is defined as any signal spike that is strictly greater than $3$ standard deviations above the mean. Create a Boolean mask tensor (filled with `True` and `False`) of shape `(1000, 5)` that flags these anomalies.

**Hint:** PyTorch handles broadcasting beautifully. To find anomalies, your logic will look something like `processed_signal > (means + 3 * stds)`.

In [ ]:
torch.manual_seed(42)
raw_signal = torch.randn(100,5)
processed_signal = raw_signal.clone()

k = torch.max(processed_signal)
pos = torch.argmax(processed_signal)

mean_sig = torch.mean(processed_signal,dim=0)
std_sig = torch.std(processed_signal,dim=0)
processed_signal[10,2] = 50
mask = processed_signal>(mean_sig+3*std_sig)
processed_signal[mask]

tensor([50.])

---

### Level 2: The Covariance Engine (Advanced Linear Algebra & Dimensionality)
You are building the risk-assessment module for a quantitative trading algorithm. You need to calculate the Mahalanobis distance, which measures how far a set of stock returns is from the average, factoring in how the stocks correlate with each other.

**The Scenario:**
You are analyzing 3 tech stocks. You need to use matrix inversion and determinant checks to calculate portfolio variance.

**Your Mission:**
1. **The Covariance Matrix:** Create a $3 \times 3$ matrix $C$. To ensure it represents valid, realistic correlations (symmetric and positive semi-definite), generate a random $3 \times 3$ matrix $A$ and compute $C = A \cdot A^T$ (where $A^T$ is the transpose of $A$).
2. **The Validity Check:** Calculate the determinant of $C$ using `torch.linalg.det()` (or `torch.det()`). If the determinant is 0, the matrix is singular and cannot be inverted.
3. **The Inverse:** Calculate the inverse matrix $C^{-1}$ using `torch.inverse()`.
4. **The Returns Vector:** Create a 1D tensor of simulated daily returns $R$ with shape `(3,)`.
5. **The Shape Shifter:** To perform the final matrix multiplication, $R$ needs to be a column vector. Use `.unsqueeze()` to change $R$ from shape `(3,)` to `(3, 1)`.
6. **The Distance Calculation:** Compute the scalar risk metric:
   $$Risk = R^T \cdot C^{-1} \cdot R$$
   *Note: Because of how matrix math works, your final result will be a `(1, 1)` tensor. Use `.squeeze()` to crush it back down to a 0D scalar tensor.*

**Hint:** Pay close attention to `torch.transpose()`. It requires you to specify exactly which two dimensions you want to swap (e.g., `dim0=0, dim1=1`).

In [43]:
A = torch.randn(3,3)
C =  torch.matmul(A,torch.transpose(A,0,1))
Det = torch.det(C)

if Det!=0:
  C_inv = torch.inverse(C)
else:
  print("Matrix in not invertible")
R = torch.randn(3,)
R.unsqueeze_(1)
R_T = torch.transpose(R,0,1);
Temp = torch.matmul(R_T,C_inv)
Risk = torch.matmul(Temp,R)
torch.squeeze(Risk)

tensor(37.9905)


## Level 3: The Rating Quantizer (Type Casting, "Like" Tensors, & Clamping)
You are building the data ingestion pipeline for an e-commerce recommendation engine. The data science team gave you continuous, unbounded predictions, but your database only accepts integer ratings strictly between 1 and 5.

**The Scenario:**
You have a batch of raw, messy user preference scores.

**Your Mission:**
1. **The Raw Scores:** Generate a tensor `predictions` of shape `(10, 5)` using `torch.randn()`. Multiply it by 5 so the values are wildly spread out (e.g., -12.4, 8.9, 3.2).
2. **The Blank Canvas:** Create a new tensor called `final_ratings` that is the exact same shape and device as `predictions`, but filled entirely with zeros. Use the `_like()` family of functions.
3. **The Boundary Enforcement:** Apply `torch.clamp()` to the `predictions` tensor to brutally force all numbers to sit between `1.0` and `5.0`.
4. **The Quantization Experiment:** * Try `torch.round()`, `torch.floor()`, and `torch.ceil()` on your clamped predictions. Observe how they handle floats like `3.5`.
   * Pick your favorite rounding method and apply it.
5. **The Final Cast:** Your tensor is still full of floats (e.g., `4.0`, `2.0`). Cast the data type of the tensor to a 32-bit integer (`torch.int32`) so it is ready for the database.

**Hint:** The `.to()` method isn't just for moving data to the GPU; it is also the cleanest way to cast data types (e.g., `.to(torch.int32)`).

In [68]:
torch.manual_seed(42)
predictions = torch.randn(10,5)*5
final_rating = torch.zeros_like(predictions)
predictions.clamp_(1,5)
predictions.round_()
predictions.to(torch.int32)

tensor([[5, 5, 5, 1, 3],
        [1, 1, 1, 1, 5],
        [1, 1, 1, 1, 1],
        [4, 5, 1, 1, 2],
        [1, 5, 4, 5, 5],
        [5, 3, 5, 1, 1],
        [1, 4, 1, 1, 1],
        [3, 1, 5, 1, 1],
        [1, 1, 1, 3, 1],
        [5, 1, 5, 1, 4]], dtype=torch.int32)